# Contrast Curves for Cross-Processed Film

Cross-processing distorts the characteristic (H&D) curve of each color layer by a *different* amount. This notebook plots density vs. log exposure for R/G/B and shows how the toe and shoulder diverge — the origin of the 'cross-processed contrast' and natural split-tone.

See `references/color_film_layers.md` for the physical background.


In [ ]:
# Make the tools/ package importable from the notebooks/ folder.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "tools")))
import numpy as np
import matplotlib.pyplot as plt
import xpro_core as xc
print("presets:", xc.list_presets())


## A simple per-channel characteristic curve model

We approximate each layer's curve with an S-curve whose contrast (gamma) and pivot differ per channel under cross-processing.

In [ ]:
logE = np.linspace(0, 1, 256)

def channel_curve(logE, gamma, pivot):
    return xc.s_curve(logE, gamma, pivot)

# 'Normal' development: all channels aligned.
normal = {c: channel_curve(logE, 4.0, 0.5) for c in 'RGB'}

# Cross-processed: layers diverge (steeper, shifted pivots).
xpro = {
    'R': channel_curve(logE, 6.5, 0.40),  # red layer steep, warm highlights
    'G': channel_curve(logE, 7.5, 0.45),  # green layer steepest -> magenta/green cast
    'B': channel_curve(logE, 5.0, 0.55),  # blue layer lifted -> cyan shadows
}


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for c, col in zip('RGB', ['r','g','b']):
    ax[0].plot(logE, normal[c], col, label=c)
    ax[1].plot(logE, xpro[c], col, label=c)
ax[0].set_title('Normal development'); ax[1].set_title('Cross-processed')
for a in ax:
    a.set_xlabel('log exposure'); a.set_ylabel('density (relative)'); a.legend(); a.grid(alpha=0.3)
plt.tight_layout();


## Why this creates split-toning

Because the blue layer's curve is lifted in the toe while the red layer is steep, **shadows skew cyan/blue and highlights skew warm** — exactly the behaviour modelled in `shaders/split_tone_xpro.frag`. The channel *difference* below shows the predicted cast vs. tone.

In [ ]:
for c, col in zip('RGB', ['r','g','b']):
    plt.plot(logE, xpro[c]-normal[c], col, label=f'{c} delta')
plt.axhline(0, color='k', lw=0.5); plt.xlabel('log exposure'); plt.ylabel('density delta')
plt.title('Per-channel curve distortion (cross-process minus normal)'); plt.legend(); plt.grid(alpha=0.3);
